In [136]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

In [137]:
df= pd.read_csv("tweet_replies2.csv")

In [138]:
df.head()

,User,Text,Date
0,@BarisUnver,700 bin lira bütçeye uygun en dertsiz aile ara...,2025-12-26T06:33:05.000Z
1,@anumariiii,Aga her modele bi cevabın var ama nedense diy...,2025-12-26T20:53:14.000Z
2,@BarisUnver,"Vallahi anlamıyorum ya, verdiğin cevaplarda bi...",2025-12-26T21:10:03.000Z
3,@turistikoda,Ne fakirsin 700 ü ben bazen ayda kazanıyorum.,2025-12-26T20:57:29.000Z
4,@BarisUnver,"Allah sana daha çok versin, yapacak bir şey yo...",2025-12-26T21:12:59.000Z


In [139]:
df.tail()

,User,Text,Date
345,@urulusoy_ugur,Mitsubishi kanser 1.6 köpek balığı,2025-12-26T22:49:04.000Z
346,@gelecekicinuret,El arabası yada at arabası,2025-12-26T20:40:01.000Z
347,@ergunkrt,Edirne'den çıkınca bunları alabilirsiniz.,2025-12-26T23:23:19.000Z
348,@iamyourfather89,Pejo 301 ve Citroen c elysee 1.6 hdi veya 1.6 ...,2025-12-26T21:58:09.000Z
349,@mit57730385,Toyota Corolla,2025-12-26T22:38:29.000Z


In [140]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   User    350 non-null    object
 1   Text    350 non-null    object
 2   Date    350 non-null    object
dtypes: object(3)
memory usage: 8.3+ KB


In [141]:
df.isna().sum()

User    0
Text    0
Date    0
dtype: int64

In [142]:
df = df.drop("Date", axis = 1)

In [143]:
df.head()

,User,Text
0,@BarisUnver,700 bin lira bütçeye uygun en dertsiz aile ara...
1,@anumariiii,Aga her modele bi cevabın var ama nedense diy...
2,@BarisUnver,"Vallahi anlamıyorum ya, verdiğin cevaplarda bi..."
3,@turistikoda,Ne fakirsin 700 ü ben bazen ayda kazanıyorum.
4,@BarisUnver,"Allah sana daha çok versin, yapacak bir şey yo..."


In [144]:
i = df[df.User == "@BarisUnver"].index
df.drop(i, inplace = True)

In [145]:
df.drop("User", axis = 1, inplace = True)

In [146]:
df

,Text
1,Aga her modele bi cevabın var ama nedense diy...
3,Ne fakirsin 700 ü ben bazen ayda kazanıyorum.
5,Hyundai i20 birde tüplü oldu mu dadından yenm...
6,Clio
7,O paralara araba var mı ya
...,...
345,Mitsubishi kanser 1.6 köpek balığı
346,El arabası yada at arabası
347,Edirne'den çıkınca bunları alabilirsiniz.
348,Pejo 301 ve Citroen c elysee 1.6 hdi veya 1.6 ...


In [147]:
from transformers import pipeline

In [148]:
analyzer = pipeline("sentiment-analysis", model="savasy/bert-base-turkish-sentiment-cased")

Device set to use cpu


In [189]:
car_dict = [
    "fiat", "egea", "doblo", "fiorino",
    "renault", "clio", "megane", "taliant", "austral",
    "togg", "t10x",    "honda", "civic", "city", "cr-v",
    "toyota", "corolla", "yaris", "c-hr",
    "ford", "focus", "fiesta", "puma", "kuga", "ranger",
    "peugeot", "208", "3008", "2008", "408",
    "opel", "corsa", "astra", "mokka",
    "volkswagen", "vw", "polo", "golf", "passat", "tiguan", "t-roc",
    "hyundai", "i20", "i10", "tucson", "bayon",
    "citroen", "c3", "c4", "c5","dacia", "duster", "sandero",
    "chery", "omoda", "tiggo","mercedes", "bmw", "audi", "volvo", "kia",
    "mitsubishi", "fluence", "mazda","skoda", "courier" "accent blue"
]

In [218]:
def marka_tespit(text):
    text = text.lower()
    detected = []
    for arac in car_dict:
        if arac in text:
            detected.append(arac)
    return detected[0] if detected else None
    

In [219]:
def sentiment_analyzing(text):
    """Cümlenin duygu durumunu ve skorunu döndürür."""
    if len(text.split()) == 1:
            etiket = "pozitif"
    else:
        sonuc = analyzer(text)[0]
    # Model 'positive'/'negative' döndürür. Biz bunu Türkçe'ye çevirelim.
        etiket = "pozitif" if sonuc['label'] == 'positive' else "negatif"
    return etiket

In [220]:
df['tespit_edilen_arac'] = df['Text'].apply(marka_tespit)

In [221]:
df["class"] = df["Text"].apply(sentiment_analyzing)

In [223]:
positive_df = df[df["class"] == "pozitif"]

In [224]:
positive_df

,Text,tespit_edilen_arac,class
1,Aga her modele bi cevabın var ama nedense diy...,None,pozitif
5,Hyundai i20 birde tüplü oldu mu dadından yenm...,hyundai,pozitif
6,Clio,clio,pozitif
15,"Toyota,Honda ve Hyundainin 700 bin TL ye satıl...",honda,pozitif
18,"Kia ceed 2011 kullanıyorum; içi geniş, otomati...",kia,pozitif
...,...,...,...
339,Bu ülkede o 500.000-1.500.000 TL aralığında al...,egea,pozitif
340,"Hyundai Elantra (2010-2015), 1.6 benzin otomat...",hyundai,pozitif
341,Toyota 1.4 d alıp temizinden arkana yaslanacak...,toyota,pozitif
342,Hyundai,hyundai,pozitif


In [225]:
pos_sonuc_tablosu = positive_df["tespit_edilen_arac"].value_counts().reset_index()

In [226]:
pos_sonuc_tablosu

,tespit_edilen_arac,count
0,toyota,16
1,egea,11
2,honda,10
3,corolla,9
4,hyundai,8
5,fiat,5
6,ford,5
7,clio,4
8,renault,4
9,polo,3


In [227]:
negative_df = df[df["class"] == "negatif"]

In [228]:
negative_df

,Text,tespit_edilen_arac,class
3,Ne fakirsin 700 ü ben bazen ayda kazanıyorum.,None,negatif
7,O paralara araba var mı ya,None,negatif
8,2018-2019 Fiat Egea Easy paket 1.6 Multijet Ot...,fiat,negatif
9,"Egea, 301, celysee düşük km orta paket temiz s...",egea,negatif
10,Corolla 1.6/1.33 Fluence 1.5 dci C elyse / 30...,egea,negatif
...,...,...,...
344,Herhangi bir Corolla,corolla,negatif
345,Mitsubishi kanser 1.6 köpek balığı,mitsubishi,negatif
346,El arabası yada at arabası,None,negatif
347,Edirne'den çıkınca bunları alabilirsiniz.,None,negatif


In [229]:
neg_sonuc_tablosu = negative_df["tespit_edilen_arac"].value_counts().reset_index()

In [230]:
neg_sonuc_tablosu

,tespit_edilen_arac,count
0,egea,25
1,toyota,16
2,clio,15
3,corolla,15
4,honda,14
5,fiat,13
6,hyundai,7
7,skoda,6
8,renault,5
9,megane,5


In [231]:
df_empty = df[df["tespit_edilen_arac"].isna()].reset_index()

In [232]:
df_empty

,index,Text,tespit_edilen_arac,class
0,1,Aga her modele bi cevabın var ama nedense diy...,None,pozitif
1,3,Ne fakirsin 700 ü ben bazen ayda kazanıyorum.,None,negatif
2,7,O paralara araba var mı ya,None,negatif
3,21,manuel dizel accent blue km düşük olmasına dik...,None,pozitif
4,24,kazalı belalı çakal kasa e46,None,negatif
...,...,...,...,...
85,322,"700K araç alıyorsanız, sadede expertiz değil a...",None,negatif
86,325,courier tabiki 3 tane aldık taş gibi araba,None,pozitif
87,331,Valla dertsiz ve aile arabası diyorsan. Temiz ...,None,negatif
88,346,El arabası yada at arabası,None,negatif


In [233]:
df_empty.to_csv("emptydata.csv")

In [244]:
positive_df.reset_index(inplace=True)

In [253]:
i = positive_df[positive_df["tespit_edilen_arac"].isna()].index

In [255]:
i

Index([  0,   6,   9,  10,  13,  15,  19,  20,  29,  36,  43,  48,  52,  54,
        55,  57,  60,  64,  68,  69,  71,  73,  84,  90,  97, 100, 102, 104,
       105, 107, 108, 112, 114, 115, 116, 120, 122, 123, 124, 128],
      dtype='int64')

In [258]:
positive_df.drop(i,axis = 0,inplace=True)

C:\Users\hp\AppData\Local\Temp\ipykernel_19868\1712382625.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  positive_df.drop(i,axis = 0,inplace=True)


In [259]:
positive_df

,index,Text,tespit_edilen_arac,class
1,5,Hyundai i20 birde tüplü oldu mu dadından yenm...,hyundai,pozitif
2,6,Clio,clio,pozitif
3,15,"Toyota,Honda ve Hyundainin 700 bin TL ye satıl...",honda,pozitif
4,18,"Kia ceed 2011 kullanıyorum; içi geniş, otomati...",kia,pozitif
5,20,Ford focus sedan abi 10 senede tek arızası yok,ford,pozitif
...,...,...,...,...
133,339,Bu ülkede o 500.000-1.500.000 TL aralığında al...,egea,pozitif
134,340,"Hyundai Elantra (2010-2015), 1.6 benzin otomat...",hyundai,pozitif
135,341,Toyota 1.4 d alıp temizinden arkana yaslanacak...,toyota,pozitif
136,342,Hyundai,hyundai,pozitif


In [260]:
positive_df["tespit_edilen_arac"].value_counts()

tespit_edilen_arac
toyota        16
egea          11
honda         10
corolla        9
hyundai        8
fiat           5
ford           5
clio           4
renault        4
polo           3
fluence        3
mazda          2
citroen        2
peugeot        2
kia            1
dacia          1
volkswagen     1
c4             1
passat         1
2008           1
mitsubishi     1
taliant        1
volvo          1
doblo          1
duster         1
mercedes       1
megane         1
fiesta         1
Name: count, dtype: int64

In [262]:
valuable_car = positive_df[positive_df["tespit_edilen_arac"] == "toyota"]

In [264]:
valuable_car.reset_index()

,level_0,index,Text,tespit_edilen_arac,class
0,28,72,Toyota al reyis,toyota,pozitif
1,32,82,Toyota,toyota,pozitif
2,39,100,Toyota corolla,toyota,pozitif
3,40,101,Toyota corolla 1.33 al kafan rahat etsin,toyota,pozitif
4,42,108,Hiç düşünmeyin Toyota diyenlere sizde katılın...,toyota,pozitif
5,45,123,12-13-14 toyota corolla dertsiz tasasız,toyota,pozitif
6,56,155,Toyota corolla,toyota,pozitif
7,77,206,Kendim kullandım gerçekten toyota corolla çok ...,toyota,pozitif
8,78,208,2015 Toyota corolla,toyota,pozitif
9,79,209,Herhangi x lira bütçeye uygun dertsiz araba he...,toyota,pozitif


In [266]:
twovaluable_car = positive_df[positive_df["tespit_edilen_arac"] == "egea"]

In [267]:
twovaluable_car

,index,Text,tespit_edilen_arac,class
8,27,Sorunun cevabı belli kesinlikle Egea. Alabilec...,egea,pozitif
12,36,Egea,egea,pozitif
16,43,2015-2018 arası Egea 1.3 m.jet al. 1 milyon km...,egea,pozitif
24,63,Egea 1.6 e-torq - Aisin marka sorunsuz japon ...,egea,pozitif
26,68,6 yıl 2017 model easy (bos paket) 1.3 egea kul...,egea,pozitif
59,167,Egea,egea,pozitif
75,202,Egea al dostum. 3 ay önce finansal açıdan raha...,egea,pozitif
80,213,Corolla Için söylenecek söz yok! Ama aile işin...,egea,pozitif
89,229,Eğer cok fazla özellik otomatik istemiyorsan e...,egea,pozitif
94,242,Egea,egea,pozitif
